### Q5 (Cross-domain) — Where in NYC do housing problems and serious crashes concentrate?
**Why it matters:** If the same districts carry both a heavy housing-maintenance burden and a high number of serious crashes, the city could coordinate housing inspections and street-safety work there instead of treating them separately. The map shows both burdens on one view: bubble size = housing problems, bubble colour = serious crashes (2022–2025 totals per community district).


In [1]:
def build_q5(df_integrated, df_centroids):
    """Q5: Map of housing problems (bubble size) and serious crashes (colour) per community district."""
    # Sum both measures per district over the years present in the (possibly filtered) data
    d = (df_integrated.groupby('cd_id', as_index=False)
         [['housing_problems', 'crashes']].sum())

    # Attach each district's centre point and readable label for the map
    d = d.merge(df_centroids[['cd_id', 'cd_label', 'borough', 'latitude', 'longitude']],
                on='cd_id', how='inner')

    # Cross-domain ratio for the hover (guard against division by zero)
    d['crashes_per_10k_problems'] = (
        d['crashes'] / d['housing_problems'].replace(0, np.nan) * 10_000
    ).round(1)

    fig = px.scatter_mapbox(
        d,
        lat='latitude', lon='longitude',
        size='housing_problems', color='crashes',
        hover_name='cd_label',
        hover_data={'borough': True,
                    'housing_problems': ':,',
                    'crashes': ':,',
                    'crashes_per_10k_problems': True,
                    'latitude': False, 'longitude': False},
        color_continuous_scale='YlOrRd',
        size_max=35,
        zoom=9.5, center={'lat': 40.71, 'lon': -73.94},
        mapbox_style='open-street-map',
        title='Where housing problems and serious crashes concentrate (by community district)',
        labels={'housing_problems': 'Housing problems',
                'crashes': 'Serious crashes',
                'crashes_per_10k_problems': 'Crashes per 10k problems',
                'borough': 'Borough'},
    )
    fig.update_layout(height=600, margin=dict(l=0, r=0, t=50, b=0))
    return fig

build_q5(df_integrated, df_centroids).show()

NameError: name 'df_integrated' is not defined

**Answer:** Housing problems are concentrated in [the Bronx and northern Manhattan / central Brooklyn], led by [district A] with [N] problems over 2022–2025. Serious crashes are concentrated in [district B] ([N] crashes) and [district C]. The two burdens overlap in [district X and district Y], which appear as large, dark bubbles, so these are candidates for coordinated attention. Elsewhere they diverge: [e.g. eastern Queens districts have many crashes but relatively few housing problems]. This is an association only; it does not mean one burden causes the other.

**Limitation:** Each district is drawn as a single centre point, so the map hides variation inside a district, and raw totals favour large, populous districts (per-capita rates would need population data).

### Q12 (Transportation) — Are night crashes more severe than day crashes, and does that differ by borough?
**Why it matters:** If serious crashes at night injure or kill more people per crash, safety measures such as lighting, speed enforcement and late-night patrols could be targeted at the hours and boroughs where severity is highest. Night = 20:00–05:59, day = 06:00–19:59. Severity is measured two ways: victims per crash, and the % of crashes with at least one fatality.

In [ ]:
from plotly.subplots import make_subplots

def build_q12(df_crashes):
    """Q12: Day vs night severity of serious crashes per borough (victims/crash and % fatal)."""
    d = df_crashes.dropna(subset=['crash_hour', 'borough']).copy()

    # Night = 20:00-05:59, day = 06:00-19:59
    is_night = (d['crash_hour'] >= 20) | (d['crash_hour'] < 6)
    d['period'] = np.where(is_night, 'Night', 'Day')
    d['has_fatality'] = d['total_fatalities'] > 0

    g = (d.groupby(['borough', 'period'], as_index=False)
           .agg(crashes=('has_fatality', 'size'),
                victims=('total_victims', 'sum'),
                fatal_crashes=('has_fatality', 'sum')))
    g['victims_per_crash'] = (g['victims'] / g['crashes']).round(2)
    g['fatal_pct'] = (g['fatal_crashes'] / g['crashes'] * 100).round(1)

    boroughs = sorted(g['borough'].unique())
    colors = {'Day': '#F2A541', 'Night': '#2E3A87'}

    fig = make_subplots(rows=1, cols=2, subplot_titles=(
        'Victims per crash', '% of crashes with a fatality'))

    for period in ['Day', 'Night']:
        p = g[g['period'] == period].set_index('borough').reindex(boroughs).reset_index()
        hover = ('<b>%{x}</b> – ' + period +
                 '<br>Crashes: %{customdata[0]:,}'
                 '<br>Victims per crash: %{customdata[1]}'
                 '<br>Fatal crashes: %{customdata[2]:,} (%{customdata[3]}%)<extra></extra>')
        custom = p[['crashes', 'victims_per_crash', 'fatal_crashes', 'fatal_pct']].values
        fig.add_trace(go.Bar(x=p['borough'], y=p['victims_per_crash'], name=period,
                             marker_color=colors[period], customdata=custom,
                             hovertemplate=hover, legendgroup=period), row=1, col=1)
        fig.add_trace(go.Bar(x=p['borough'], y=p['fatal_pct'], name=period,
                             marker_color=colors[period], customdata=custom,
                             hovertemplate=hover, legendgroup=period,
                             showlegend=False), row=1, col=2)

    fig.update_layout(barmode='group', height=500,
                      title='Night vs day severity of serious crashes by borough',
                      legend_title_text='Time of day')
    fig.update_xaxes(title_text='Borough')
    fig.update_yaxes(title_text='Victims per crash', row=1, col=1)
    fig.update_yaxes(title_text='Crashes with a fatality (%)', row=1, col=2)
    return fig

build_q12(df_crashes).show()

**Answer:** Across NYC, [N]% of serious crashes happen at night. Night crashes have [higher / similar] severity: [X] victims per crash at night vs [Y] by day, and [A]% of night crashes involve a fatality vs [B]% by day. The night–day fatality gap is largest in [borough] ([A]% vs [B]%) and smallest in [borough]. Night is therefore associated with [more severe] outcomes, which supports targeting lighting and speed enforcement at night, but the data cannot show why (speed, visibility, alcohol and traffic volume are not in the dataset).

**Limitation:** Fatal crashes are rare, so fatality percentages for small groups are noisy, especially Staten Island at night with only [N] crashes. The hover shows the crash count for each bar so readers can judge reliability.